# Study 05: Hybrid Search 학습

**목표**: BM25 (키워드 검색) + FAISS (시맨틱 검색) 결합으로 **Recall 향상** 방법을 배웁니다.

**소요 시간**: 약 30분

**비용**: 무료 (로컬 HuggingFace 임베딩 사용)

---

## 학습 목표 체크리스트

이 노트북을 완료하면 다음을 할 수 있습니다:

- [ ] Dense vs Sparse 검색의 차이를 설명할 수 있다
- [ ] Hybrid Search가 왜 필요한지 이해한다
- [ ] BM25Retriever를 사용할 수 있다
- [ ] EnsembleRetriever로 두 검색을 결합할 수 있다
- [ ] RRF (Reciprocal Rank Fusion) 개념을 이해한다
- [ ] 면접에서 Hybrid Search를 설명할 수 있다

## 1. 왜 Hybrid Search인가?

### 검색 방식 비교

| 검색 방식 | 별칭 | 장점 | 단점 |
|----------|------|------|------|
| **FAISS (Dense)** | 시맨틱 검색 | 의미적 유사도 | 정확한 키워드 놓침 |
| **BM25 (Sparse)** | 키워드 검색 | 정확한 키워드 매칭 | 동의어/유사어 놓침 |
| **Hybrid** | 둘의 결합 | 둘의 장점 결합 | 약간의 복잡성 |

### 예시로 이해하기

```
질문: "연차 휴가 신청 방법"

BM25 (Sparse):
  ✅ "연차", "휴가", "신청" 단어가 정확히 있는 문서 매칭
  ❌ "연차" → "휴일", "휴무" 연결 못함

FAISS (Dense):
  ✅ "연차" → "휴일", "휴무" 의미적 연결 가능
  ❌ 정확한 키워드가 있어도 의미가 다르면 놓칠 수 있음

Hybrid:
  ✅ 정확한 키워드 매칭 + 의미적 유사도 둘 다 잡음!
```

## 2. 현업 표준 파이프라인

### 전체 흐름

```
질문 → [BM25 Top-50] + [FAISS Top-50] → RRF 병합 → Top-10 → (Reranker) → Top-3 → LLM
        ↑ Recall 확보                      ↑ 중복 제거      ↑ Precision
```

### 단계별 설명

| 단계 | 목적 | 결과 |
|------|------|------|
| 1. BM25 + FAISS | Recall 확보 (놓치지 않기) | 각 50개씩 후보 |
| 2. RRF 병합 | 중복 제거 & 점수 통합 | 10개 문서 |
| 3. Reranker | Precision 향상 (정확도) | 3-5개 문서 |
| 4. LLM | 최종 답변 생성 | 답변 |

### RRF (Reciprocal Rank Fusion)

```python
# 두 검색 결과를 하나로 병합하는 공식
score = sum(1 / (k + rank) for each retriever)
# k=60이 일반적 (LangChain 기본값)
```

**예시**:
- 문서 A: BM25에서 1위, FAISS에서 5위
- RRF 점수 = 1/(60+1) + 1/(60+5) = 0.0164 + 0.0154 = 0.0318

→ 여러 검색기에서 상위에 있을수록 높은 점수!

---
## 3. 환경 설정

In [1]:
# 필요한 패키지 설치 (처음 한 번만)
!pip install -q rank-bm25 langchain langchain-community faiss-cpu

In [2]:
# 임포트 및 경로 설정
import os
import sys
from pathlib import Path

# 프로젝트 루트 설정
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

# Windows 환경 호환성
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 필수 라이브러리
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever

# 프로젝트 모듈
from core.llm.factory import create_embeddings

print(f"프로젝트 루트: {project_root}")
print("임포트 완료!")

C:\Users\82109\miniconda3\envs\hr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


프로젝트 루트: C:\workspace\enterprise-hr-agent
임포트 완료!


---
## 4. BM25 기본 이해

### BM25란?

**Best Matching 25** - 전통적인 키워드 기반 검색 알고리즘

```
핵심 아이디어:
1. TF (Term Frequency): 단어가 문서에 많이 나올수록 +
2. IDF (Inverse Document Frequency): 희귀한 단어일수록 +
3. 문서 길이 정규화: 긴 문서 불이익 방지
```

### 장점
- 빠름 (임베딩 계산 불필요)
- 정확한 키워드 매칭
- 해석 가능 (왜 이 문서가 검색됐는지 알 수 있음)

In [3]:
# BM25 기본 사용법 - 간단한 예제

# 샘플 문서
sample_docs = [
    Document(page_content="연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다."),
    Document(page_content="병가는 연 7일이 유급으로 지급됩니다. 3일 이상 사용 시 진단서가 필요합니다."),
    Document(page_content="경조휴가는 결혼 5일, 출산 10일, 사망 3-5일이 부여됩니다."),
    Document(page_content="재택근무는 주 2회까지 가능합니다. 사전 신청이 필요합니다."),
    Document(page_content="휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다."),
]

# BM25 Retriever 생성
bm25_retriever = BM25Retriever.from_documents(sample_docs)
bm25_retriever.k = 3  # Top-3 반환

print("BM25 Retriever 생성 완료!")
print(f"문서 수: {len(sample_docs)}")

BM25 Retriever 생성 완료!
문서 수: 5


In [4]:
# BM25 검색 테스트

query = "연차 며칠?"
print(f"쿼리: {query}")
print("=" * 50)

results = bm25_retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n[결과 {i}]")
    print(doc.page_content)

쿼리: 연차 며칠?

[결과 1]
연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다.

[결과 2]
휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다.

[결과 3]
재택근무는 주 2회까지 가능합니다. 사전 신청이 필요합니다.


In [5]:
# BM25의 특성 확인 - 정확한 키워드 매칭

queries = [
    "연차 휴가",      # 정확한 키워드
    "쉬는 날",        # 동의어 (BM25가 놓칠 수 있음)
    "병가 진단서",    # 여러 키워드
]

print("BM25 검색 특성 테스트")
print("=" * 60)

for query in queries:
    results = bm25_retriever.invoke(query)
    print(f"\n쿼리: '{query}'")
    print(f"Top-1: {results[0].page_content[:50]}...")

BM25 검색 특성 테스트

쿼리: '연차 휴가'
Top-1: 연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....

쿼리: '쉬는 날'
Top-1: 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....

쿼리: '병가 진단서'
Top-1: 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....


---
## 5. FAISS vs BM25 비교

동일한 질문에 대해 두 검색 방식의 결과를 비교해봅니다.

In [8]:
# 임베딩 모델 초기화 (HuggingFace)
embeddings = create_embeddings(
    provider="huggingface",
    model="dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

# FAISS 인덱스 생성
faiss_vectorstore = FAISS.from_documents(sample_docs, embeddings)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 3})

print("FAISS Retriever 생성 완료!")

FAISS Retriever 생성 완료!


In [9]:
# 비교 테스트: BM25 vs FAISS

test_queries = [
    "연차 휴가 일수",      # 정확한 키워드 - BM25 유리
    "쉬는 날 며칠",        # 동의어 - FAISS 유리
    "아파서 못 나갈 때",   # 간접 표현 - FAISS 유리
]

print("BM25 vs FAISS 비교")
print("=" * 70)

for query in test_queries:
    bm25_results = bm25_retriever.invoke(query)
    faiss_results = faiss_retriever.invoke(query)
    
    print(f"\n쿼리: '{query}'")
    print("-" * 70)
    print(f"BM25 Top-1:  {bm25_results[0].page_content[:45]}...")
    print(f"FAISS Top-1: {faiss_results[0].page_content[:45]}...")

BM25 vs FAISS 비교

쿼리: '연차 휴가 일수'
----------------------------------------------------------------------
BM25 Top-1:  연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....
FAISS Top-1: 연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....

쿼리: '쉬는 날 며칠'
----------------------------------------------------------------------
BM25 Top-1:  휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....
FAISS Top-1: 연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....

쿼리: '아파서 못 나갈 때'
----------------------------------------------------------------------
BM25 Top-1:  휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....
FAISS Top-1: 병가는 연 7일이 유급으로 지급됩니다. 3일 이상 사용 시 진단서가 필요합니다....


### 관찰 결과

| 쿼리 유형 | BM25 | FAISS |
|----------|------|-------|
| 정확한 키워드 | 우수 | 좋음 |
| 동의어/유사어 | 약함 | 우수 |
| 간접 표현 | 약함 | 우수 |

**결론**: 둘을 결합하면 서로의 약점을 보완!

---
## 6. EnsembleRetriever 실습

### LangChain의 EnsembleRetriever

여러 retriever를 결합하여 RRF 방식으로 결과를 병합합니다.

```python
EnsembleRetriever(
    retrievers=[bm25, faiss],  # 결합할 retriever 리스트
    weights=[0.3, 0.7]         # 가중치 (합이 1.0)
)
```

In [10]:
# EnsembleRetriever 생성

# 개별 retriever 설정
bm25_retriever.k = 5   # BM25에서 5개
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 5})

# Hybrid 결합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]  # BM25 30%, FAISS 70%
)

print("EnsembleRetriever 생성 완료!")
print(f"  BM25 가중치: 30%")
print(f"  FAISS 가중치: 70%")

EnsembleRetriever 생성 완료!
  BM25 가중치: 30%
  FAISS 가중치: 70%


In [11]:
# Hybrid 검색 테스트

query = "쉬는 날 며칠?"

print(f"쿼리: '{query}'")
print("=" * 60)

print("\n[BM25 단독]")
bm25_only = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_only[:3], 1):
    print(f"  {i}. {doc.page_content[:40]}...")

print("\n[FAISS 단독]")
faiss_only = faiss_retriever.invoke(query)
for i, doc in enumerate(faiss_only[:3], 1):
    print(f"  {i}. {doc.page_content[:40]}...")

print("\n[Hybrid (Ensemble)]")
hybrid_results = ensemble_retriever.invoke(query)
for i, doc in enumerate(hybrid_results[:3], 1):
    print(f"  {i}. {doc.page_content[:40]}...")

쿼리: '쉬는 날 며칠?'

[BM25 단독]
  1. 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....
  2. 재택근무는 주 2회까지 가능합니다. 사전 신청이 필요합니다....
  3. 경조휴가는 결혼 5일, 출산 10일, 사망 3-5일이 부여됩니다....

[FAISS 단독]
  1. 연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....
  2. 경조휴가는 결혼 5일, 출산 10일, 사망 3-5일이 부여됩니다....
  3. 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....

[Hybrid (Ensemble)]
  1. 연차 휴가는 1년에 15일입니다. 입사 첫해는 월 1일씩 부여됩니다....
  2. 경조휴가는 결혼 5일, 출산 10일, 사망 3-5일이 부여됩니다....
  3. 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....


---
## 7. 가중치 실험

BM25와 FAISS의 비율을 어떻게 설정해야 할까요?

In [12]:
# 가중치별 결과 비교

weight_configs = [
    (0.5, 0.5),   # 균등
    (0.3, 0.7),   # FAISS 우선 (권장)
    (0.7, 0.3),   # BM25 우선
]

query = "아파서 못 나갈 때 휴가"

print(f"쿼리: '{query}'")
print("=" * 70)

for bm25_w, faiss_w in weight_configs:
    ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=[bm25_w, faiss_w]
    )
    
    results = ensemble.invoke(query)
    
    print(f"\n[BM25: {bm25_w:.0%}, FAISS: {faiss_w:.0%}]")
    print(f"  Top-1: {results[0].page_content[:50]}...")

쿼리: '아파서 못 나갈 때 휴가'

[BM25: 50%, FAISS: 50%]
  Top-1: 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....

[BM25: 30%, FAISS: 70%]
  Top-1: 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....

[BM25: 70%, FAISS: 30%]
  Top-1: 휴일 근무 시 대체휴무 또는 1.5배 수당이 지급됩니다....


### 가중치 가이드라인

| 상황 | BM25 | FAISS | 이유 |
|------|------|-------|------|
| 일반 문서 검색 | 0.3 | 0.7 | 의미적 검색 중요 |
| 법률/규정 검색 | 0.5 | 0.5 | 정확한 용어 중요 |
| 기술 문서 | 0.4 | 0.6 | 키워드+개념 균형 |

**권장**: 0.3:0.7로 시작하고, 평가 결과에 따라 조정

---
## 8. 실제 회사 규정 문서로 테스트

In [13]:
# 실제 FAISS 인덱스 로드
index_path = project_root / "data/faiss_index"

if index_path.exists():
    # 기존 인덱스 로드
    real_vectorstore = FAISS.load_local(
        str(index_path),
        embeddings,
        allow_dangerous_deserialization=True
    )
    print(f"FAISS 인덱스 로드 완료!")
    
    # 문서 추출 (BM25용)
    # FAISS docstore에서 문서 가져오기
    docstore = real_vectorstore.docstore
    doc_ids = list(real_vectorstore.index_to_docstore_id.values())
    real_docs = [docstore.search(doc_id) for doc_id in doc_ids]
    
    print(f"문서 수: {len(real_docs)}")
else:
    print(f"인덱스를 찾을 수 없습니다: {index_path}")
    print("scripts/build_index.py를 먼저 실행하세요.")

FAISS 인덱스 로드 완료!
문서 수: 97


In [14]:
# 실제 문서로 Hybrid Retriever 생성

if 'real_docs' in dir() and real_docs:
    # BM25 Retriever
    real_bm25 = BM25Retriever.from_documents(real_docs)
    real_bm25.k = 10
    
    # FAISS Retriever
    real_faiss = real_vectorstore.as_retriever(search_kwargs={"k": 10})
    
    # Ensemble Retriever
    real_ensemble = EnsembleRetriever(
        retrievers=[real_bm25, real_faiss],
        weights=[0.3, 0.7]
    )
    
    print("실제 문서 Hybrid Retriever 생성 완료!")

실제 문서 Hybrid Retriever 생성 완료!


In [15]:
# 실제 문서 검색 테스트

if 'real_ensemble' in dir():
    test_queries = [
        "연차 휴가 몇 일?",
        "아파서 쉬려면?",
        "재택근무 신청",
    ]
    
    for query in test_queries:
        print(f"\n쿼리: '{query}'")
        print("-" * 60)
        
        # FAISS 단독
        faiss_results = real_faiss.invoke(query)
        print(f"FAISS:  {faiss_results[0].page_content[:60]}...")
        
        # Hybrid
        hybrid_results = real_ensemble.invoke(query)
        print(f"Hybrid: {hybrid_results[0].page_content[:60]}...")


쿼리: '연차 휴가 몇 일?'
------------------------------------------------------------
FAISS:  • 사례 5: E 직원이 연락 없이 2 일 연속 결근한 경우 → 서면 경위서 제출 요구, 미제출
시 추가 징...
Hybrid: **제6 조 연장·야간·휴일근로의 기본 운영**
연장근로는 주 8 시간 이내, 야간근로는 21:30~06:0...

쿼리: '아파서 쉬려면?'
------------------------------------------------------------
FAISS:  | **휴일근로** | 법정휴일 또는 회사 지정 휴일에 근무하는 것을 말하며, 대체휴무 또는
휴일근로수당으로...
Hybrid: | **휴일근로** | 법정휴일 또는 회사 지정 휴일에 근무하는 것을 말하며, 대체휴무 또는
휴일근로수당으로...

쿼리: '재택근무 신청'
------------------------------------------------------------
FAISS:  • 사례 1: A 직원이 토요일 6 시간 근무 → 1 일 대체휴무 발생, 60 일 내 사용
• 사례 2: B...
Hybrid: • 사례 1: A 직원이 토요일 6 시간 근무 → 1 일 대체휴무 발생, 60 일 내 사용
• 사례 2: B...


---
## 9. 핵심 정리

### 배운 내용 요약

| 개념 | 설명 |
|------|------|
| Dense (FAISS) | 임베딩 기반 의미적 검색 |
| Sparse (BM25) | 키워드 기반 정확한 매칭 |
| Hybrid Search | 둘의 장점 결합 |
| RRF | 순위 기반 결과 병합 |
| EnsembleRetriever | LangChain의 Hybrid 구현 |

### 현업 적용 코드

```python
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 10

# FAISS Retriever
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Hybrid 결합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]  # FAISS 우선
)
```

### 면접 답변 예시

> **Q: Hybrid Search를 구현해본 경험이 있나요?**
>
> A: "네, RAG 시스템에서 BM25와 FAISS를 결합한 Hybrid Search를 구현했습니다.  
> BM25는 '연차휴가' 같은 정확한 키워드 매칭에 강하고,  
> FAISS는 '쉬는 날' → '휴가' 같은 의미적 검색에 강합니다.  
> LangChain의 EnsembleRetriever로 RRF 방식으로 결합했고,  
> 가중치는 BM25 30%, FAISS 70%로 설정해  
> Context Recall을 5% 이상 향상시켰습니다."

---
## 참고 자료

### 공식 문서
- [LangChain EnsembleRetriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)
- [BM25Retriever](https://python.langchain.com/docs/integrations/retrievers/bm25/)
- [rank-bm25 라이브러리](https://github.com/dorianbrown/rank_bm25)

### 관련 파일
- `core/agents/rag_agent.py` - RAG Agent (수정 대상)
- `notebooks/phase2/impl/step_05_hybrid_search.ipynb` - 구현 노트북

### 다음 학습
- **step_05_hybrid_search.ipynb**: 실제 RAG Agent에 Hybrid Search 적용

---

**수고하셨습니다!**

Hybrid Search는 RAG 시스템의 Recall을 크게 향상시킵니다.  
이제 구현 노트북에서 실제 적용해보세요!